[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_05_Agent_Memory.ipynb)

# 🧠 Lesson 5: Agent Memory
### *From Goldfish to Elephant — Teaching Your Agent to Remember*

---

## Where We Are

In Lesson 4 you built a **ReAct agent** — a loop that thinks, calls tools, observes results, and repeats. That agent was powerful, but it had a critical flaw:

> Every time a new conversation started, it remembered **nothing** from before.

You asked it "What's the weather in London?" and it answered. You asked again five minutes later — same question, same blank slate, same API call. It had no memory of your preferences, your history, or your name.

**Memory is what separates a one-shot tool from an intelligent assistant.**

---

## What You'll Build Today

We'll implement **four types of memory**, each solving a different problem:

| Memory Type | What It Solves | Analogy |
|---|---|---|
| **In-Context (Short-term)** | Keep conversation history alive | RAM — fast, limited |
| **Sliding Window** | Prevent context overflow | RAM with auto-eviction |
| **Summarization** | Compress history intelligently | Cliff Notes of RAM |
| **Long-term (File/DB)** | Survive restarts, cross sessions | Hard drive |
| **Episodic** | Recall specific past events | Diary |

By the end, you'll have a single agent that uses **all four in combination**.

---

## The Core Problem: LLMs Are Stateless

This is the most important thing to internalize before we write any code:

```
Every API call to Claude is completely independent.
Claude has NO memory of previous calls by default.
```

When you call `client.messages.create(...)`, Claude sees only what's in the `messages` list you pass right now. Nothing else. Memory is entirely **your responsibility as the developer**.

The "memory" you see in Claude.ai chat? That's just Claude.ai maintaining a `messages` list and passing the whole thing on every call. There's no magic — just concatenated history.

Let's start building.

## 🔧 Setup

In [ ]:
# Install dependencies
!pip install anthropic tiktoken -q

# tiktoken is OpenAI's tokenizer — we'll use it for token counting
# It doesn't count Claude tokens exactly, but it's close enough for learning
print("✅ Dependencies installed")

In [ ]:
import anthropic
import json
import os
import time
import datetime
from pathlib import Path

# Load API key from Colab Secrets
# In Colab: left sidebar → 🔑 Secrets → Add ANTHROPIC_API_KEY
try:
    from google.colab import userdata
    api_key = userdata.get('ANTHROPIC_API_KEY')
    print("✅ API key loaded from Colab Secrets")
except Exception:
    # Fallback for local Jupyter
    api_key = os.environ.get('ANTHROPIC_API_KEY')
    if api_key:
        print("✅ API key loaded from environment variable")
    else:
        raise ValueError("❌ No API key found. Add ANTHROPIC_API_KEY to Colab Secrets!")

client = anthropic.Anthropic(api_key=api_key)
MODEL = "claude-haiku-4-5-20251001"  # Fast and cheap — perfect for learning
print(f"✅ Client ready, using {MODEL}")

---

## Part 1: The Problem — No Memory by Default

Let's first demonstrate the problem clearly. We'll make two separate API calls and show that the model has zero memory between them.

In [ ]:
# ─── CALL 1: Tell it your name ───
response1 = client.messages.create(
    model=MODEL,
    max_tokens=100,
    messages=[
        {"role": "user", "content": "Hi! My name is Gourav. Please remember this."}
    ]
)
print("Call 1 response:", response1.content[0].text)

print("\n" + "─" * 50 + "\n")

# ─── CALL 2: Ask it your name — completely new messages list ───
response2 = client.messages.create(
    model=MODEL,
    max_tokens=100,
    messages=[
        {"role": "user", "content": "What is my name?"}
    ]
)
print("Call 2 response:", response2.content[0].text)
print("\n💡 It doesn't know your name — each call is a clean slate!")

---

## Part 2: In-Context Memory (Short-term)

The simplest fix: **keep appending messages to a list and pass the whole list every time**.

This is exactly what every chat interface does. The "conversation" is just a list that grows with each turn.

```
Turn 1:  messages = [ {user: "Hi, I'm Gourav"} ]
Turn 2:  messages = [ {user: "Hi, I'm Gourav"}, {assistant: "Nice to meet you!"}, {user: "What's my name?"} ]
Turn 3:  messages = [ ...all of the above..., {assistant: "Gourav!"}, {user: "..."} ]
```

Let's build a `ConversationBuffer` class:

In [ ]:
class ConversationBuffer:
    """
    The simplest possible memory: a growing list of messages.
    Every message (user + assistant) gets appended and sent on the next call.
    """

    def __init__(self, system_prompt: str = "You are a helpful assistant."):
        self.system_prompt = system_prompt
        self.messages = []  # ← This is the memory

    def chat(self, user_message: str) -> str:
        # Step 1: Append user message to memory
        self.messages.append({"role": "user", "content": user_message})

        # Step 2: Send the ENTIRE history to the API
        response = client.messages.create(
            model=MODEL,
            max_tokens=300,
            system=self.system_prompt,
            messages=self.messages  # ← Full history every time
        )

        # Step 3: Get the assistant reply
        assistant_reply = response.content[0].text

        # Step 4: Append assistant reply to memory (so next turn includes it)
        self.messages.append({"role": "assistant", "content": assistant_reply})

        return assistant_reply

    def show_history(self):
        print(f"\n📜 Conversation History ({len(self.messages)} messages):")
        for i, msg in enumerate(self.messages):
            role_icon = "👤" if msg["role"] == "user" else "🤖"
            print(f"  [{i+1}] {role_icon} {msg['role'].upper()}: {msg['content'][:80]}..." 
                  if len(msg['content']) > 80 else 
                  f"  [{i+1}] {role_icon} {msg['role'].upper()}: {msg['content']}")


# ─── Test it ───
bot = ConversationBuffer(system_prompt="You are a friendly assistant who remembers what users tell you.")

print("Turn 1:", bot.chat("Hi! My name is Gourav and I'm a Java developer learning AI."))
print("\nTurn 2:", bot.chat("What is my name?"))
print("\nTurn 3:", bot.chat("What language do I come from?"))

bot.show_history()

# 💡 EXPERIMENT: Try asking follow-up questions that require memory:
# bot.chat("Based on my background, what should I learn next in AI?")
# bot.chat("What did I tell you in my very first message?")

### The Problem with In-Context Memory: Context Window Limits

Claude Haiku's context window is 200,000 tokens. That sounds huge, but it adds up fast:
- A 1-hour support conversation: ~10,000 tokens
- A coding session: ~50,000–100,000 tokens  
- Cost also scales with context length

**Rule of thumb:** Any production agent needs a strategy for managing context window growth.

Let's add token counting to see how fast we burn through the window:

In [ ]:
import tiktoken

def count_tokens(messages: list, system: str = "") -> int:
    """
    Rough token count for a messages list.
    Uses GPT-4's tokenizer as a proxy (Claude's tokenizer is similar).
    Not exact, but useful for learning.
    """
    enc = tiktoken.get_encoding("cl100k_base")  # GPT-4 encoding
    total = len(enc.encode(system))
    for msg in messages:
        total += len(enc.encode(msg["content"]))
        total += 4  # ~4 tokens per message for role/formatting overhead
    return total


class ConversationBufferWithTokenCount(ConversationBuffer):
    """Adds token counting to show context growth."""

    def chat(self, user_message: str) -> str:
        reply = super().chat(user_message)
        tokens = count_tokens(self.messages, self.system_prompt)
        print(f"  📊 Context window used: ~{tokens:,} tokens")
        return reply


# Simulate a growing conversation
bot2 = ConversationBufferWithTokenCount(
    system_prompt="You are a verbose assistant. Always give detailed 3-sentence answers."
)

questions = [
    "What is machine learning?",
    "How does gradient descent work?",
    "What is a neural network?",
    "Explain backpropagation.",
]

for q in questions:
    print(f"\n👤 User: {q}")
    reply = bot2.chat(q)
    print(f"🤖 Bot: {reply[:100]}...")

# 💡 EXPERIMENT: What happens to token count over 20, 50, 100 turns?
# The number just keeps growing. That's the problem we solve next.

---

## Part 3: Sliding Window Memory

The simplest approach to managing context growth: **only keep the last N messages**.

Think of it like a sliding window. As new messages arrive, old ones fall off the back.

```
Window size = 4:

Turn 1:  [msg1]
Turn 2:  [msg1, msg2]
Turn 3:  [msg1, msg2, msg3]
Turn 4:  [msg1, msg2, msg3, msg4]   ← window full
Turn 5:  [msg2, msg3, msg4, msg5]   ← msg1 dropped!
Turn 6:  [msg3, msg4, msg5, msg6]   ← msg2 dropped!
```

**Tradeoff:** Simple and cheap, but the agent forgets things from early in the conversation.

In [ ]:
class SlidingWindowMemory:
    """
    Keeps only the last `window_size` messages.
    Old messages are permanently discarded.
    """

    def __init__(self, system_prompt: str = "You are a helpful assistant.", window_size: int = 6):
        self.system_prompt = system_prompt
        self.window_size = window_size  # Keep last N messages
        self.messages = []
        self.total_messages_seen = 0

    def chat(self, user_message: str) -> str:
        # Add user message
        self.messages.append({"role": "user", "content": user_message})
        self.total_messages_seen += 1

        # Trim: keep only last `window_size` messages
        # Important: we must keep pairs (user+assistant) to avoid API errors
        # So we trim to even numbers, keeping user-assistant pairs
        if len(self.messages) > self.window_size:
            trimmed = len(self.messages) - self.window_size
            self.messages = self.messages[trimmed:]

        # API call with trimmed window
        response = client.messages.create(
            model=MODEL,
            max_tokens=200,
            system=self.system_prompt,
            messages=self.messages
        )
        reply = response.content[0].text
        self.messages.append({"role": "assistant", "content": reply})

        # Report window status
        tokens = count_tokens(self.messages, self.system_prompt)
        print(f"  📊 Window: {len(self.messages)}/{self.window_size} msgs | ~{tokens} tokens | Total seen: {self.total_messages_seen}")

        return reply


# Test: The window should cap at 6 messages
print("=== Sliding Window Demo (window=6) ===\n")
bot3 = SlidingWindowMemory(window_size=6)

turns = [
    "My name is Gourav.",
    "I love building systems.",
    "My favorite color is blue.",
    "I work at a fintech startup.",
    "What is my name?"  # Should remember (within window)
]

for turn in turns:
    print(f"\n👤 {turn}")
    reply = bot3.chat(turn)
    print(f"🤖 {reply}")

print("\n\n=== Now let's overflow the window ===\n")
# Add many more messages to push early ones out
overflow_turns = [
    "Tell me about Python.",
    "Explain lists vs tuples.",
    "What is my name?",  # Will it still remember?
]
for turn in overflow_turns:
    print(f"\n👤 {turn}")
    reply = bot3.chat(turn)
    print(f"🤖 {reply}")

# 💡 EXPERIMENT: Change window_size to 2. Now what gets forgotten?
# 💡 EXPERIMENT: What if you set window_size = 100? It becomes ConversationBuffer.

---

## Part 4: Summarization Memory

The sliding window is blunt — it throws away information. What if old messages contain important facts (your name, your goals, your preferences)?

**Summarization memory** is smarter: instead of discarding old messages, it **uses the LLM to compress them** into a summary. The summary takes up far fewer tokens but preserves key information.

```
Before summarization (8 messages, ~1000 tokens):
  "Hi I'm Gourav" / "Nice to meet you Gourav!"
  "I'm a Java dev" / "That's great!"
  "I work at a fintech" / "Interesting!"
  "I want to learn AI agents" / "Awesome goal!"

After summarization (1 summary + current msgs, ~200 tokens):
  SUMMARY: "User is Gourav, a Java developer at a fintech company. Goal: learn AI agents."
  + last 2 messages
```

This is the approach used by production chatbots like ChatGPT.

In [ ]:
class SummarizationMemory:
    """
    Keeps recent messages in full detail.
    When history gets too long, compresses old messages into a summary
    using the LLM itself.
    """

    def __init__(
        self,
        system_prompt: str = "You are a helpful assistant.",
        max_messages_before_summary: int = 6  # Summarize when history exceeds this
    ):
        self.system_prompt = system_prompt
        self.max_messages = max_messages_before_summary
        self.messages = []       # Recent messages (kept in full)
        self.summary = ""        # Compressed history of older messages
        self.summary_count = 0   # Track how many times we've summarized

    def _build_system_prompt(self) -> str:
        """Inject summary into system prompt so the model 'remembers' old context."""
        if self.summary:
            return f"""{self.system_prompt}

## Conversation History Summary
The following is a summary of the conversation so far (before the current messages):
{self.summary}
"""
        return self.system_prompt

    def _summarize(self):
        """Compress the oldest half of messages into a running summary."""
        self.summary_count += 1
        print(f"  🔄 [Summarizing... (summary #{self.summary_count})]")

        # Take the oldest half of messages for summarization
        split = len(self.messages) // 2
        to_summarize = self.messages[:split]
        self.messages = self.messages[split:]  # Keep the newer half

        # Format the old messages as a block of text
        conversation_text = "\n".join(
            f"{m['role'].upper()}: {m['content']}" for m in to_summarize
        )

        # Build the summarization prompt
        # Include any existing summary so we're doing incremental updates
        existing = f"Previous summary: {self.summary}\n\n" if self.summary else ""
        prompt = f"""Create a concise summary of this conversation that captures key facts, 
names, preferences, and important information. Be specific and factual.

{existing}New conversation to summarize:
{conversation_text}

Summary (be concise, keep key facts):"""

        # Call LLM to generate the summary
        response = client.messages.create(
            model=MODEL,
            max_tokens=300,
            messages=[{"role": "user", "content": prompt}]
        )
        self.summary = response.content[0].text
        print(f"  📝 New summary: {self.summary[:120]}..." if len(self.summary) > 120 else f"  📝 Summary: {self.summary}")

    def chat(self, user_message: str) -> str:
        # Check if we need to summarize before adding new message
        if len(self.messages) >= self.max_messages:
            self._summarize()

        # Add user message
        self.messages.append({"role": "user", "content": user_message})

        # Call API with summary injected into system prompt
        response = client.messages.create(
            model=MODEL,
            max_tokens=300,
            system=self._build_system_prompt(),  # ← Summary lives here
            messages=self.messages
        )
        reply = response.content[0].text
        self.messages.append({"role": "assistant", "content": reply})

        tokens = count_tokens(self.messages, self._build_system_prompt())
        print(f"  📊 Recent msgs: {len(self.messages)} | ~{tokens} tokens | Has summary: {'Yes' if self.summary else 'No'}")

        return reply


# Test: Have a long conversation and watch summarization kick in
print("=== Summarization Memory Demo ===\n")
bot4 = SummarizationMemory(max_messages_before_summary=4)

conversation = [
    "My name is Gourav and I'm learning AI.",
    "I come from a Java background but I'm open to Python.",
    "My goal is to build an open source AI agent.",
    "I particularly like building systems that solve real problems.",
    "What programming languages might be useful for AI agents?",  # Triggers summarization
    "Given my background, what should be my first open-source project?",
    "What is my name and goal again?",  # Test if summary preserved key facts
]

for turn in conversation:
    print(f"\n👤 {turn}")
    reply = bot4.chat(turn)
    print(f"🤖 {reply[:150]}..." if len(reply) > 150 else f"🤖 {reply}")

# 💡 EXPERIMENT: Set max_messages_before_summary=2 to watch it summarize more aggressively
# 💡 EXPERIMENT: After the loop, do: print("\nFinal summary:", bot4.summary)

---

## Part 5: Long-term Memory (File Persistence)

All memory types so far live in RAM. Kill the Python process and all memory is gone.

**Long-term memory** persists to disk (or a database). The agent can:
- Pick up where it left off after a restart
- Remember user preferences across sessions
- Build a permanent knowledge base

In production, this could be:
- A JSON file (simple, like we'll build now)
- SQLite / PostgreSQL (structured queries)
- A vector database (semantic search — we'll cover this in Lesson 7!)

Let's build a simple file-based persistent memory:

In [ ]:
class LongTermFileMemory:
    """
    Persists agent memory to a JSON file.
    Survives Python restarts — the agent remembers you across sessions.

    Structure stored on disk:
    {
        "user_facts": {"name": "Gourav", "goal": "..."},
        "conversation_summary": "...",
        "recent_messages": [...],
        "session_count": 3,
        "last_seen": "2026-05-04T10:30:00"
    }
    """

    def __init__(self, memory_file: str = "agent_memory.json"):
        self.memory_file = Path(memory_file)
        self.memory = self._load()
        self.memory["session_count"] = self.memory.get("session_count", 0) + 1
        print(f"📂 Memory loaded. Session #{self.memory['session_count']}")
        if self.memory.get("conversation_summary"):
            print(f"📜 Existing summary: {self.memory['conversation_summary'][:100]}...")

    def _load(self) -> dict:
        """Load memory from disk, or create fresh if file doesn't exist."""
        if self.memory_file.exists():
            with open(self.memory_file, "r") as f:
                return json.load(f)
        return {
            "user_facts": {},
            "conversation_summary": "",
            "recent_messages": [],
            "session_count": 0,
            "last_seen": None
        }

    def save(self):
        """Persist current memory to disk."""
        self.memory["last_seen"] = datetime.datetime.now().isoformat()
        with open(self.memory_file, "w") as f:
            json.dump(self.memory, f, indent=2)

    def store_fact(self, key: str, value: str):
        """Explicitly store a key-value fact about the user."""
        self.memory["user_facts"][key] = value
        self.save()
        print(f"  💾 Stored: {key} = {value}")

    def get_fact(self, key: str) -> str:
        return self.memory["user_facts"].get(key, "Unknown")

    def add_message(self, role: str, content: str):
        """Add a message and keep only the last 10 in recent memory."""
        self.memory["recent_messages"].append({"role": role, "content": content})
        if len(self.memory["recent_messages"]) > 10:
            self.memory["recent_messages"] = self.memory["recent_messages"][-10:]

    def build_context_prompt(self) -> str:
        """Build a rich system prompt from all stored memory."""
        parts = ["You are a personalized assistant with memory of past interactions."]

        if self.memory["user_facts"]:
            facts_str = "\n".join(f"  - {k}: {v}" for k, v in self.memory["user_facts"].items())
            parts.append(f"\n## Known Facts About This User:\n{facts_str}")

        if self.memory["conversation_summary"]:
            parts.append(f"\n## Past Conversation Summary:\n{self.memory['conversation_summary']}")

        parts.append(f"\n## Session Info:\n  - This is session #{self.memory['session_count']}")
        if self.memory["last_seen"]:
            parts.append(f"  - Last seen: {self.memory['last_seen']}")

        return "\n".join(parts)

    def chat(self, user_message: str) -> str:
        self.add_message("user", user_message)

        response = client.messages.create(
            model=MODEL,
            max_tokens=300,
            system=self.build_context_prompt(),
            messages=self.memory["recent_messages"]
        )
        reply = response.content[0].text
        self.add_message("assistant", reply)
        self.save()  # Persist after every turn
        return reply


# ─── First Session ───
print("=" * 50)
print("SESSION 1: First time meeting the agent")
print("=" * 50)

mem1 = LongTermFileMemory("./gourav_memory.json")
mem1.store_fact("name", "Gourav")
mem1.store_fact("background", "Java developer")
mem1.store_fact("goal", "Build open-source AI agent")

r = mem1.chat("Hi! What do you know about me?")
print(f"\n🤖 {r}")

r = mem1.chat("Great! I prefer concise answers, always 2 sentences max.")
mem1.store_fact("preference", "Concise answers, max 2 sentences")
print(f"\n🤖 {r}")

print("\n" + "=" * 50)
print("SESSION 2: Coming back after a 'restart'")
print("=" * 50)

# Create a NEW instance — this simulates a restart
# It reads from disk, so all facts are still there!
mem2 = LongTermFileMemory("./gourav_memory.json")
r = mem2.chat("Hey, do you remember me?")
print(f"\n🤖 {r}")

r = mem2.chat("And what's my preference for answers?")
print(f"\n🤖 {r}")

print("\n✅ Memory persisted across 'sessions'!")
print(f"📂 Memory file saved at: {Path('./gourav_memory.json').absolute()}")

# 💡 EXPERIMENT: Run this cell again — notice session count increments
# 💡 EXPERIMENT: print(json.dumps(mem2.memory, indent=2)) to see raw stored memory

---

## Part 6: Episodic Memory

So far we've stored **facts** and **summaries**. But sometimes you need to remember **specific events** in detail:

- "The last time you asked me to fix this bug was on March 15th"
- "You ran out of budget on your last project after 3 days"
- "The best plan we found for your goal was X"

**Episodic memory** stores discrete timestamped events — like a diary. You can search through episodes and inject the relevant ones into context.

This is the foundation of what frameworks like **MemGPT** (now Letta) are built on.

In [ ]:
class EpisodicMemory:
    """
    Stores and retrieves discrete timestamped episodes.
    Each episode is a structured event with tags for filtering.

    Episode format:
    {
        "id": 1,
        "timestamp": "2026-05-04T10:30:00",
        "event": "User asked about Python decorators",
        "context": "Learning session on Python advanced features",
        "outcome": "Explained with 3 examples, user understood",
        "tags": ["python", "learning", "decorators"]
    }
    """

    def __init__(self, storage_file: str = "episodes.json"):
        self.storage_file = Path(storage_file)
        self.episodes = self._load()

    def _load(self) -> list:
        if self.storage_file.exists():
            with open(self.storage_file, "r") as f:
                return json.load(f)
        return []

    def _save(self):
        with open(self.storage_file, "w") as f:
            json.dump(self.episodes, f, indent=2)

    def record(self, event: str, context: str = "", outcome: str = "", tags: list = None):
        """Store a new episode."""
        episode = {
            "id": len(self.episodes) + 1,
            "timestamp": datetime.datetime.now().isoformat(),
            "event": event,
            "context": context,
            "outcome": outcome,
            "tags": tags or []
        }
        self.episodes.append(episode)
        self._save()
        print(f"  📔 Episode #{episode['id']} recorded: {event[:60]}")
        return episode

    def search(self, keyword: str = None, tag: str = None, last_n: int = None) -> list:
        """Search episodes by keyword, tag, or recency."""
        results = self.episodes

        if keyword:
            kw = keyword.lower()
            results = [e for e in results if
                       kw in e["event"].lower() or
                       kw in e["context"].lower() or
                       kw in e["outcome"].lower()]

        if tag:
            results = [e for e in results if tag.lower() in [t.lower() for t in e["tags"]]]

        if last_n:
            results = results[-last_n:]

        return results

    def format_for_prompt(self, episodes: list) -> str:
        """Format episodes as context for the LLM system prompt."""
        if not episodes:
            return "No relevant past episodes found."
        lines = []
        for e in episodes:
            ts = e['timestamp'][:10]  # Just the date
            lines.append(f"  [{ts}] {e['event']}")
            if e['outcome']:
                lines.append(f"           → Outcome: {e['outcome']}")
        return "\n".join(lines)

    def stats(self):
        print(f"\n📊 Episodic Memory Stats:")
        print(f"  Total episodes: {len(self.episodes)}")
        if self.episodes:
            all_tags = [t for e in self.episodes for t in e["tags"]]
            unique_tags = set(all_tags)
            print(f"  Unique tags: {unique_tags}")
            print(f"  First episode: {self.episodes[0]['timestamp'][:10]}")
            print(f"  Latest episode: {self.episodes[-1]['timestamp'][:10]}")


# ─── Simulate a series of learning sessions being recorded ───
episodic = EpisodicMemory("./gourav_episodes.json")

print("Recording learning episodes...\n")

episodic.record(
    event="Completed Lesson 1: LLM Fundamentals",
    context="First lesson in AI learning journey",
    outcome="Successfully called Claude API, understood tokens and temperature",
    tags=["lesson", "llm", "fundamentals", "api"]
)

episodic.record(
    event="Struggled with tool use in Lesson 3",
    context="Building a multi-tool agent",
    outcome="Resolved by understanding the tool loop cycle: detect → call → observe → respond",
    tags=["lesson", "tools", "struggle", "resolved"]
)

episodic.record(
    event="Successfully built first ReAct agent in Lesson 4",
    context="Implemented think/act/observe loop with calculator and weather tools",
    outcome="Agent solved multi-step math problem with 3 tool calls",
    tags=["lesson", "agent", "react", "milestone"]
)

episodic.record(
    event="User asked about production deployment",
    context="Curiosity question during Lesson 4",
    outcome="Noted: cover observability, rate limits, cost optimization in Lesson 8",
    tags=["production", "question", "curriculum"]
)

# ─── Search and use episodic memory ───
print("\n\n=== Searching Episodic Memory ===\n")

# Search for struggles
struggles = episodic.search(tag="struggle")
print("Past struggles:")
print(episodic.format_for_prompt(struggles))

# Search for milestones
milestones = episodic.search(tag="milestone")
print("\nMilestones achieved:")
print(episodic.format_for_prompt(milestones))

# Search for lessons
lessons = episodic.search(tag="lesson")
print("\nAll lesson episodes:")
print(episodic.format_for_prompt(lessons))

episodic.stats()

# ─── Now use episodic memory in an LLM call ───
print("\n\n=== Using Episodic Memory in LLM Call ===\n")

relevant_episodes = episodic.search(tag="lesson")
episode_context = episodic.format_for_prompt(relevant_episodes)

system = f"""You are a personalized AI tutor for Gourav.

## Gourav's Learning History:
{episode_context}

Use this history to give personalized, context-aware advice."""

response = client.messages.create(
    model=MODEL,
    max_tokens=300,
    system=system,
    messages=[{"role": "user", "content": "Based on my learning history, what should I focus on this week?"}]
)
print("🤖 Personalized advice based on episodic memory:")
print(response.content[0].text)

# 💡 EXPERIMENT: Add an episode about a specific mistake you made
# Then ask: "What mistakes should I avoid when building agents?"

---

## Part 7: The Complete Memory-Aware Agent

Now let's combine all four memory types into a single production-ready agent.

This is the architecture used in real personal assistant systems:

```
┌─────────────────────────────────────────────────────────────┐
│                     MEMORY-AWARE AGENT                      │
│                                                             │
│  ┌────────────────┐    ┌─────────────────┐                 │
│  │   User Facts   │    │   Episodes DB   │   (Long-term)   │
│  │ (file-backed)  │    │  (file-backed)  │                 │
│  └───────┬────────┘    └────────┬────────┘                 │
│          │                     │                           │
│          ▼                     ▼                           │
│  ┌─────────────────────────────────────────┐               │
│  │         System Prompt Builder           │               │
│  │  (facts + summary + relevant episodes)  │               │
│  └──────────────────────┬──────────────────┘               │
│                         │                                  │
│                         ▼                                  │
│  ┌──────────────────────────────────────┐                  │
│  │  Recent Messages (Sliding Window)    │  (Short-term)    │
│  └──────────────────────────────────────┘                  │
│                         │                                  │
│                         ▼                                  │
│               [ API Call to Claude ]                       │
│                         │                                  │
│                         ▼                                  │
│  ┌──────────────────────────────────────┐                  │
│  │   Auto-extract & store new facts     │                  │
│  │   Record episode if significant      │                  │
│  └──────────────────────────────────────┘                  │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
class FullMemoryAgent:
    """
    A complete memory-aware agent combining:
    1. Long-term fact storage (persistent JSON)
    2. Episodic memory (structured past events)
    3. Sliding window for recent messages
    4. Auto fact extraction from conversations
    """

    def __init__(self, user_name: str, memory_dir: str = "./memory"):
        self.user_name = user_name
        Path(memory_dir).mkdir(exist_ok=True)

        # Initialize all memory layers
        self.facts = LongTermFileMemory(f"{memory_dir}/{user_name}_facts.json")
        self.episodes = EpisodicMemory(f"{memory_dir}/{user_name}_episodes.json")
        self.recent_messages = []   # Sliding window
        self.window_size = 8        # Keep last 8 messages in context

        print(f"\n🤖 Agent initialized for {user_name}")
        print(f"   Facts on file: {len(self.facts.memory.get('user_facts', {}))}")
        print(f"   Episodes on file: {len(self.episodes.episodes)}")

    def _build_system_prompt(self, query: str) -> str:
        """Build a rich system prompt from all memory layers."""
        parts = [f"You are a personalized assistant for {self.user_name}."]

        # Layer 1: User facts
        facts = self.facts.memory.get("user_facts", {})
        if facts:
            facts_str = "\n".join(f"  - {k}: {v}" for k, v in facts.items())
            parts.append(f"\n## Known Facts:\n{facts_str}")

        # Layer 2: Relevant episodic memories
        # In a real system you'd use semantic search here (Lesson 7!)
        # For now, use the 3 most recent episodes
        recent_episodes = self.episodes.search(last_n=3)
        if recent_episodes:
            episode_text = self.episodes.format_for_prompt(recent_episodes)
            parts.append(f"\n## Recent Events:\n{episode_text}")

        # Layer 3: Session info
        session = self.facts.memory.get('session_count', 1)
        parts.append(f"\n## Session: #{session}")
        parts.append("\nBe personalized, concise, and use what you know about the user.")

        return "\n".join(parts)

    def _auto_extract_facts(self, user_message: str, assistant_reply: str):
        """
        Ask the LLM to extract any new facts from the conversation turn.
        This is 'self-directed memory' — the agent decides what to remember.
        """
        extraction_prompt = f"""Extract any important facts about the user from this exchange.
Return ONLY a JSON object with key-value facts, or {{}} if nothing important.
Examples of good facts: name, job, goal, preference, struggle, skill level.

User said: {user_message}
Assistant said: {assistant_reply[:200]}

JSON facts:"""

        try:
            response = client.messages.create(
                model=MODEL,
                max_tokens=150,
                messages=[{"role": "user", "content": extraction_prompt}]
            )
            facts_text = response.content[0].text.strip()
            # Extract JSON from the response
            start = facts_text.find("{")
            end = facts_text.rfind("}") + 1
            if start >= 0 and end > start:
                extracted = json.loads(facts_text[start:end])
                for key, value in extracted.items():
                    if value and key not in ["name"]:  # Don't overwrite name
                        self.facts.store_fact(key, str(value))
        except Exception:
            pass  # Auto-extraction is best-effort, don't crash if it fails

    def chat(self, user_message: str, record_as_episode: bool = False) -> str:
        """Send a message with full memory context."""
        # Add to sliding window
        self.recent_messages.append({"role": "user", "content": user_message})
        if len(self.recent_messages) > self.window_size:
            self.recent_messages = self.recent_messages[-self.window_size:]

        # Build rich system prompt from all memory layers
        system = self._build_system_prompt(user_message)

        # Call API
        response = client.messages.create(
            model=MODEL,
            max_tokens=400,
            system=system,
            messages=self.recent_messages
        )
        reply = response.content[0].text

        # Add reply to sliding window
        self.recent_messages.append({"role": "assistant", "content": reply})

        # Auto-extract and store any new facts (background operation)
        self._auto_extract_facts(user_message, reply)

        # Record as episode if flagged as significant
        if record_as_episode:
            self.episodes.record(
                event=f"User asked: {user_message[:80]}",
                outcome=f"Agent responded about: {reply[:80]}",
                tags=["conversation"]
            )

        # Persist facts
        self.facts.save()

        return reply

    def remember_milestone(self, event: str, outcome: str, tags: list = None):
        """Manually record a significant event."""
        self.episodes.record(event=event, outcome=outcome, tags=tags or ["milestone"])


# ─── Demonstration ───
print("=" * 60)
print("FULL MEMORY AGENT — Complete Demo")
print("=" * 60)

agent = FullMemoryAgent("Gourav", memory_dir="./memory")

# Seed some facts
agent.facts.store_fact("name", "Gourav")
agent.facts.store_fact("background", "Java developer")
agent.facts.store_fact("goal", "Build open-source AI agent")
agent.facts.store_fact("learning_stage", "Lesson 5 of 9 — Agent Memory")

# Seed some episodes
agent.remember_milestone(
    "Completed ReAct agent in Lesson 4",
    "Built working agent with tool loop, calculator and weather tools",
    tags=["lesson", "milestone", "agent"]
)

# Have a conversation
print("\n" + "─" * 40)
conversations = [
    ("Hi! What do you know about me and where I am in my learning journey?", True),
    ("I'm finding context window management confusing. Any tips?", False),
    ("What should be my next step after today's lesson?", True),
]

for msg, record_episode in conversations:
    print(f"\n👤 Gourav: {msg}")
    reply = agent.chat(msg, record_as_episode=record_episode)
    print(f"\n🤖 Agent: {reply}")
    print("─" * 40)

print("\n✅ All memory layers active!")
print(f"   Stored facts: {len(agent.facts.memory.get('user_facts', {}))}")
print(f"   Recorded episodes: {len(agent.episodes.episodes)}")
print(f"   Recent messages in window: {len(agent.recent_messages)}")

# 💡 EXPERIMENT: Restart the kernel and run from this cell only
#                Watch how the agent remembers you from the saved files!
# 💡 EXPERIMENT: Ask "What mistakes am I making?" — auto-extracted facts will inform this
# 💡 EXPERIMENT: Add agent.remember_milestone() for your own achievements and ask for a review

---

## Part 8: Memory Architecture Decision Guide

A quick mental model for choosing the right memory strategy in real projects:

In [ ]:
# ─── Decision Guide ───
print("""
╔══════════════════════════════════════════════════════════════╗
║           MEMORY ARCHITECTURE DECISION GUIDE                 ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  Q: How long is your typical conversation?                   ║
║                                                              ║
║  < 10 turns  → ConversationBuffer (simple, no overhead)     ║
║  10–50 turns → SlidingWindow or SummarizationMemory         ║
║  Unlimited   → SummarizationMemory + LongTermMemory         ║
║                                                              ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  Q: Does the agent need to remember across sessions?         ║
║                                                              ║
║  No  → Keep everything in-memory (ConversationBuffer)       ║
║  Yes → LongTermFileMemory or a database                     ║
║                                                              ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  Q: Does the agent need to search past events?              ║
║                                                              ║
║  Simple keyword → EpisodicMemory (what we built)            ║
║  Semantic/fuzzy → Vector DB + Embeddings (Lesson 7!)        ║
║                                                              ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  Production stack for a personal assistant:                  ║
║    - Sliding window (recent)                                 ║
║    - Summarization (medium term)                             ║
║    - PostgreSQL / SQLite (facts + preferences)               ║
║    - Vector DB (episodic + semantic search)                  ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")

# ─── Token cost comparison across strategies ───
print("Token Cost Simulation (50-turn conversation):")
print()

# Assume each message ~50 tokens
msg_tokens = 50
turns = 50

# ConversationBuffer: grows linearly
buffer_total = sum(i * msg_tokens for i in range(1, turns + 1))

# SlidingWindow (size 10): constant after window fills
sliding_total = (sum(range(1, 10)) + 10 * (turns - 9)) * msg_tokens

# Summarization: ~200 token summary + 10 recent msgs
summary_per_call = 200 + (10 * msg_tokens)
summarization_total = summary_per_call * turns

# Cost: Claude Haiku input = ~$0.25 per 1M tokens
cost_per_token = 0.25 / 1_000_000

strategies = [
    ("ConversationBuffer", buffer_total),
    ("SlidingWindow(10)", sliding_total),
    ("SummarizationMemory", summarization_total),
]

for name, total_tokens in strategies:
    cost = total_tokens * cost_per_token
    print(f"  {name:<25} {total_tokens:>8,} tokens  ~${cost:.4f}")

print()
print("💡 In a 50-turn conversation, sliding window uses ~3x fewer tokens than full buffer!")

---

## Summary: What You Built Today

You went from a stateless LLM to a full memory-aware agent. Here's what you now know:

| Concept | What It Does | When To Use |
|---|---|---|
| **ConversationBuffer** | Appends all messages, sends full history | Short conversations |
| **SlidingWindow** | Keeps last N messages, drops older ones | Medium conversations, predictable token cost |
| **SummarizationMemory** | LLM compresses old context into summary | Long conversations, key facts must survive |
| **LongTermFileMemory** | Persists facts to disk/DB, survives restarts | Personal assistants, multi-session agents |
| **EpisodicMemory** | Stores discrete timestamped events | Diary-style recall, learning systems |

### Key Mental Model

> **Memory is just careful engineering of what goes into the `messages` list and `system` prompt on each API call.**
> There is no magic. Claude sees only what you explicitly give it.

---

## What's Next: Lesson 6 — Multi-Agent Systems

You now have a single agent that remembers. In the next lesson:

- **Orchestrators** — agents that direct other agents
- **Subagents** — specialized agents for specific tasks
- **Handoff patterns** — how agents pass work to each other
- **Parallel execution** — running multiple agents simultaneously

Think of it like this: a single agent is one person. Multiple agents are a team. Teams solve harder problems.

---

## Bonus: Connecting to Production Frameworks

Everything you built today manually is what production frameworks automate:

- **LangChain** — `ConversationBufferMemory`, `ConversationSummaryMemory`, `VectorStoreRetrieverMemory`
- **LlamaIndex** — `ChatMemoryBuffer`, `VectorMemory`
- **MemGPT / Letta** — Full OS-like memory management with virtual context
- **Mem0** — Drop-in memory layer for any agent

Now that you understand the fundamentals, those framework docs will make complete sense!


In [ ]:
# ─── Final Checkpoint: Test Your Understanding ───
print("=== Lesson 5 Self-Check ===")
print()
print("Try answering these without looking:")
print()
questions = [
    "1. Why is an LLM considered 'stateless'?",
    "2. What's the difference between sliding window and summarization memory?",
    "3. When would you choose episodic memory over a conversation summary?",
    "4. What goes into the 'messages' list vs the 'system' prompt?",
    "5. How would you combine all memory types for a coding assistant?",
]
for q in questions:
    print(q)

print()
print("Next lesson: Multi-Agent Systems 🤖🤖🤖")
print()
print("Keep building. See you in Lesson 6!")